In [1]:
import scanpy as sc
import numpy as np
import pandas as pd

In [2]:
file_path = "../data/training_cells.h5ad"

In [3]:
adata = sc.read_h5ad(file_path)

In [4]:
adata

AnnData object with n_obs × n_vars = 17882 × 19226
    obs: 'nCount_RNA', 'nFeature_RNA', 'percent.mt', 'sgrna_id', 'sgrna_symbol', 'channel'
    var: 'features'

## Preprocessing by the instruction on Kaggle:
> We start from UMI counts of 19,226 genes. The gene list is defined by Gencode v.46.
Raw counts are first log-normalized:
>
> * Divide the UMI counts in each cell by the total UMI count in that cell
> * Multiply by 10,000
> * Log-transform with log1p (log(x + 1)); logarithm base = 2.
> 
> Normalization is performed on all 19,226 genes to facilitate compatibility with other pre-processed datasets. The normalized data are then subset to the 5,127 genes relevant for the challenge. Finally, expression values for each gene are averaged per perturbation. We use a simple arithmetic mean, disregarding batch information.

In [5]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata, base=2)

### Filtering the genes to predict

In [6]:
submission_path = "../data/sample_submission.csv"

In [7]:
submission = pd.read_csv(submission_path)

In [8]:
target_genes = list(submission.columns)[1:]
adata_target = adata[:, adata.var_names.isin(target_genes)].copy()
adata_target

AnnData object with n_obs × n_vars = 17882 × 5127
    obs: 'nCount_RNA', 'nFeature_RNA', 'percent.mt', 'sgrna_id', 'sgrna_symbol', 'channel'
    var: 'features'
    uns: 'log1p'

In [9]:
adata_target.to_df().head(5)

,A1BG,A1CF,AADAC,AAK1,AARS1,AASS,ABCA1,ABCA12,ABCA5,ABCB5,...,ZP3,ZPBP,ZRANB3,ZSCAN18,ZSCAN31,ZSWIM5,ZSWIM6,ZSWIM7,ZWINT,ZYX
AAACCAAAGACGCGAA_ch_1,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,...,1.150484,0.0,0.687009,0.0,0.000000,0.0,0.687009,1.150484,0.687009,0.687009
AAACCAAAGCAAATGA_ch_1,1.415037,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,...,0.000000,0.0,0.000000,0.0,0.000000,0.0,2.369234,0.874469,0.000000,0.000000
AAACCAAAGCAGTCTA_ch_1,0.627569,0.0,0.0,0.627569,0.447110,0.0,0.000000,0.0,0.0,0.0,...,0.240806,0.0,1.397736,0.0,0.000000,0.0,0.787944,0.627569,0.932262,0.627569
AAACCAAAGGGCATAG_ch_1,0.000000,0.0,0.0,1.340110,0.595092,0.0,1.605151,0.0,0.0,0.0,...,1.015168,0.0,1.340110,0.0,0.000000,0.0,1.340110,0.000000,0.595092,1.015168
AAACCATTCCAATCGA_ch_1,0.616832,0.0,0.0,0.616832,1.648013,0.0,0.000000,0.0,0.0,0.0,...,0.000000,0.0,0.000000,0.0,0.616832,0.0,2.070741,0.616832,0.000000,0.616832


## Grouping by perturbation

In [10]:
df = adata_target.to_df()
df['sgrna_symbol'] = adata_target.obs['sgrna_symbol']

### Get the average values
Just like the given `train_data_means.csv` file, make a CSV file with average expression values
> Average expression values of unperturbed cells (non-targeting sgRNA) and cells with 80 selected perturbations. These cells are from the same experiment as the cells in the validation and test datasets. The last row, containing non-targeting in the pert_symbol column, is the baseline expression used for reference. All submissions should contain delta expression relative to this row.

In [11]:
grouped_mean = df.groupby('sgrna_symbol').mean()
print(grouped_mean)

                   A1BG      A1CF     AADAC      AAK1     AARS1      AASS  \
sgrna_symbol                                                                
ACLY           0.311642  0.022102  0.100949  0.378671  0.715857  0.000000   
ALDOA          0.627445  0.043961  0.094335  0.442062  0.658125  0.000000   
APAF1          0.467114  0.030714  0.090910  0.470155  0.732770  0.002820   
ARID2          0.517185  0.018776  0.102561  0.445692  0.687079  0.010236   
BAG1           0.409021  0.034862  0.129969  0.462849  0.819877  0.002999   
...                 ...       ...       ...       ...       ...       ...   
TGFBR2         0.537884  0.015554  0.106994  0.455972  0.723697  0.005621   
USP22          0.443085  0.044874  0.123302  0.288553  0.675256  0.000000   
VEGFA          0.498489  0.009164  0.096523  0.420647  0.753090  0.003990   
WAC            0.481894  0.023404  0.085533  0.476920  0.801459  0.000000   
non-targeting  0.474062  0.025797  0.093244  0.443709  0.737890  0.002591   

In [12]:
grouped_mean.to_csv("../data/train_grouped_mean.csv")